In [ ]:
import socket, struct, json, os, requests

sock = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
sock.connect(f"{os.path.expanduser('~')}/Library/Caches/runt/runtimed.sock")

# Handshake: identify as Blob channel
handshake = json.dumps({"channel": "blob"}).encode()
sock.sendall(struct.pack(">I", len(handshake)) + handshake)

# Send store request
req = json.dumps({"action": "store", "media_type": "text/plain"}).encode()
sock.sendall(struct.pack(">I", len(req)) + req)

# Send the actual blob data as the next frame
data = b"Hello from the blob store!"
sock.sendall(struct.pack(">I", len(data)) + data)

# Read response
length = struct.unpack(">I", sock.recv(4))[0]
resp = json.loads(sock.recv(length))
print(json.dumps(resp, indent=2))
sock.close()

In [ ]:
requests.get(f"http://127.0.0.1:59799/blob/{resp['hash']}").text

In [ ]:
!ls ~/Library/Caches/runt/blobs

!ls ~/Library/Caches/runt/blobs/{resp['hash'][:2]}

In [ ]:
!cat ~/Library/Caches/runt/blobs/{resp['hash'][:2]}/{resp['hash'][2:]}.meta | jq